#### **Step 1: Import Required Libraries**


In [9]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp, row_number, split)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 11, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [10]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/oracle/raw/user_preferences"
df_raw_user_preferences = spark.read.format("parquet").load(raw_transaction_path)
display(df_raw_user_preferences.limit(10))

StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c552cf26-5180-425d-a1f3-e0def3ccb5ae)

### **Column-by-Column Transformation Plan**
user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the user_preferences dataframe for data quality checks**

In [11]:
# Select specific columns and count nulls
null_counts = df_raw_user_preferences.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['user_id','favorite_sports', 'odds_format', 'marketing_opt_in']
    ]
)

# Display results
null_counts.show()

StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 13, Finished, Available, Finished)

+-------+---------------+-----------+----------------+
|user_id|favorite_sports|odds_format|marketing_opt_in|
+-------+---------------+-----------+----------------+
|      0|              0|       7506|            4182|
+-------+---------------+-----------+----------------+



#### **Step 4: Data Transformation to favorite_sports column**

Issues:
- Inconsistent Casing, Sports names appear in inconsistent cases (e.g., "cricket", "BOXING", "Golf"), which can affect grouping and analytics.
- Duplicate Entries Within a Single Row, Some users have repeated sports in the same entry (e.g., "Boxing,Boxing" or "Football,Basketball,Boxing,Boxing"), introducing noise.
- Irregular Formatting, Sports lists have varying whitespace and inconsistent separators (e.g., extra spaces before/after commas).
- Lack of Standardization for Sorting, Sports appear in arbitrary order, which can complicate comparison and deduplication during analysis.

Transformations:
- Splitting, Used split() to separate comma-delimited strings into arrays.
- Trimming and Casing, Applied trim() to remove leading/trailing whitespace and initcap() to standardize each sport to title case (e.g., "cricket" → "Cricket").
- Removing Duplicates, Applied array_distinct() to remove repeated sports within each user’s list.
- Alphabetical Sorting, Used sort_array() to sort sports names consistently for easier comparison and analytics.
- Rejoining to String, Used concat_ws(", ") to convert cleaned arrays back into user-readable comma-separated strings.


In [12]:
# Step 1: Check distinct values before cleaning
# df_raw_user_preferences.select("favorite_sports").distinct().show(30, truncate=False)

# Step 2: Normalize and transform favorite_sports values
df_cleaned_user_preferences = df_raw_user_preferences.withColumn(
    "favorite_sports",
    sort_array(  # Alphabetical sorting
        array_distinct(  # Remove duplicates
            transform(
                split(trim(col("favorite_sports")), ","),  # Split comma-separated string
                lambda sport: initcap(trim(sport))  # Trim and convert to title case
            )
        )
    )
)

# Step 3: Convert array back to comma-separated string for easier consumption
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "favorite_sports",
    concat_ws(", ", col("favorite_sports"))
)

# Step 4: Display cleaned distinct values
df_cleaned_user_preferences.select("favorite_sports").distinct().show(30, truncate=False)


StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 14, Finished, Available, Finished)

+--------------------------------------+
|favorite_sports                       |
+--------------------------------------+
|Baseball, Boxing, Cricket, Rugby      |
|Boxing, Cricket, Golf                 |
|Cricket, Hockey                       |
|Football, Golf, Rugby                 |
|Tennis                                |
|Hockey, Rugby                         |
|Baseball, Boxing, Tennis              |
|Boxing                                |
|Boxing, Football                      |
|Boxing, Football, Tennis              |
|Boxing, Hockey, Rugby                 |
|Baseball, Golf, Hockey, Tennis        |
|Basketball, Rugby, Tennis             |
|Cricket, Football, Golf, Hockey       |
|Baseball, Cricket, Football, Tennis   |
|Football, Golf, Rugby, Tennis         |
|Basketball, Cricket, Football, Tennis |
|Baseball, Basketball, Boxing, Football|
|Basketball, Golf, Rugby               |
|Baseball, Basketball, Boxing, Rugby   |
|Golf                                  |
|Hockey, Rugby, 

#### **Step 5: Data Transformation to odds_format column**
Issues:
- Inconsistent Casing, Values like "DECIMAL", "fractional", "AMERICAN" appear in various cases.
- Non-standard Variants, Variants like "Dec" for "Decimal" reduce consistency and may lead to data grouping issues.
- Empty or Null Values, 1,251 records have nulls (None or empty string), which can mislead analytics if not handled explicitly.

Transformations:
- Standardize Casing: Normalize to title case.
- Map Variants: Convert "DECIMAL", "Dec" → "Decimal", "fractional" → "Fractional", etc.
- Null Cleanup: Convert empty strings ("") to None.

In [13]:
from pyspark.sql.functions import col, lower, trim, when

# Define a mapping dictionary for standardizing values
odds_format_mapping = {
    "decimal": "Decimal",
    "dec": "Decimal",
    "fractional": "Fractional",
    "american": "American"
}

# Normalize and clean odds_format column
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "odds_format",
    when(trim(col("odds_format")) == "", None)  # Treat empty strings as null
    .otherwise(
        when(
            lower(trim(col("odds_format"))).isin(list(odds_format_mapping.keys())),
            # Map to standardized values
            when(lower(trim(col("odds_format"))) == "decimal", "Decimal")
            .when(lower(trim(col("odds_format"))) == "dec", "Decimal")
            .when(lower(trim(col("odds_format"))) == "fractional", "Fractional")
            .when(lower(trim(col("odds_format"))) == "american", "American")
        ).otherwise("Unknown")  # Handle any unrecognized non-null value
    )
)

# Show distinct cleaned values
df_cleaned_user_preferences.select("odds_format").distinct().show()


StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 15, Finished, Available, Finished)

+-----------+
|odds_format|
+-----------+
|    Unknown|
| Fractional|
|    Decimal|
|   American|
+-----------+



#### **Step 6: Data Transformation to marketing_opt_in column**
Issues:
- Mixed boolean representations ('Yes', 'TRUE', '0', etc.)
- Casing inconsistencies
- Missing values

Transformations:
- Normalize to lowercase
- Map truthy values (true, yes, 1) → True
- Map falsy values (false, no, 0) → False
- Replace unclear/missing values → None

In [14]:
# Preview distinct raw values (optional)
# df_cleaned_user_preferences.select("marketing_opt_in").distinct().show()

# Define the cleaning transformation
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "marketing_opt_in",
    when(
        lower(col("marketing_opt_in")).isin("true", "yes", "1"),
        True
    ).when(
        lower(col("marketing_opt_in")).isin("false", "no", "0"),
        False
    ).otherwise(None)
)

# Preview distinct cleaned values
df_cleaned_user_preferences.select("marketing_opt_in").distinct().show()

# Optional: Display a few records
display(df_cleaned_user_preferences.limit(10))


StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 16, Finished, Available, Finished)

+----------------+
|marketing_opt_in|
+----------------+
|            true|
|           false|
|            NULL|
+----------------+



SynapseWidget(Synapse.DataFrame, f9cbd71f-8e1b-466d-b163-a452cb2a305d)

#### **Step 7: Deduplication of user_id in df_cleaned_user_preferences**
Issues:

- Multiple entries with the same user_id
- Conflicting or duplicate updates for the same target row during merge

Transformations:

- Partitioned the dataset by user_id
- Ordered records by load_timestamp in descending order
- Kept only the latest record per user using row_number()
- Dropped helper columns after filtering to clean the dataset

In [15]:
# Add ingestion_date
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "ingestion_date", current_date()
)

# Add load_timestamp FIRST
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "load_timestamp", lit(load_ts)
)

# Deduplicate based on user_id, keeping latest by load_timestamp
window_spec = Window.partitionBy("user_id").orderBy(col("load_timestamp").desc())

df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "row_num", row_number().over(window_spec)
).filter(col("row_num") == 1).drop("row_num")

StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 17, Finished, Available, Finished)

#### **Step 7: Data Quality Checks**
| Check                      | Description                                            |
| -------------------------- | ------------------------------------------------------ |
| `null_favorite_sports`     | Ensures every user has at least one favorite sport     |
| `invalid_marketing_opt_in` | Ensures only `True` or `False` allowed                 |
| `empty_odds_format`        | Flags blank odds formats, but allows `NULL`            |
| `duplicate_user_ids`       | Ensures no duplicate users in the preferences table    |
| `min_expected_rows`        | Ensures there are at least 1000 rows (can be adjusted) |


In [16]:
from pyspark.sql.functions import col, count, trim, when, lit
from pyspark.sql import DataFrame

# ---------- 1. Define expected values ----------
valid_marketing_opt_in = [True, False]
min_expected_rows = 1000  # Optional: minimum row count expectation

# ---------- 2. Run DQ checks ----------
dq_metrics = (
    df_cleaned_user_preferences
    .select(
        when(col("favorite_sports").isNull() | (trim(col("favorite_sports")) == ""), 1).otherwise(0).alias("null_favorite_sports"),
        when(~col("marketing_opt_in").isin(valid_marketing_opt_in) & col("marketing_opt_in").isNotNull(), 1).otherwise(0).alias("invalid_marketing_opt_in"),
        when(trim(col("odds_format")) == "", 1).otherwise(0).alias("empty_odds_format")
    )
)

# ---------- 3. Aggregate metrics ----------
aggregated_metrics = dq_metrics.groupBy().sum().collect()[0]

null_favorite_sports     = aggregated_metrics["sum(null_favorite_sports)"]
invalid_marketing_opt_in = aggregated_metrics["sum(invalid_marketing_opt_in)"]
empty_odds_format        = aggregated_metrics["sum(empty_odds_format)"]

# ---------- 4. Additional DQ checks ----------
total_rows         = df_cleaned_user_preferences.count()
duplicate_user_ids = df_cleaned_user_preferences.groupBy("user_id").count().filter("count > 1").count()

# ---------- 5. Set thresholds ----------
thresholds = {
    "null_favorite_sports": 10,
    "invalid_marketing_opt_in": 0,
    "empty_odds_format": 50,         # lenient because nulls are allowed
    "duplicate_user_ids": 0,
    "min_rows": min_expected_rows
}

# ---------- 6. Determine status ----------
dq_passed = (
    null_favorite_sports     <= thresholds["null_favorite_sports"]
    and invalid_marketing_opt_in <= thresholds["invalid_marketing_opt_in"]
    and empty_odds_format    <= thresholds["empty_odds_format"]
    and duplicate_user_ids   <= thresholds["duplicate_user_ids"]
    and total_rows           >= thresholds["min_rows"]
)

# ---------- 7. Print Summary ----------
print("\n Data Quality Report:")
print(f"  • null_favorite_sports        = {null_favorite_sports}")
print(f"  • invalid_marketing_opt_in    = {invalid_marketing_opt_in}")
print(f"  • empty_odds_format           = {empty_odds_format}")
print(f"  • duplicate_user_ids          = {duplicate_user_ids}")
print(f"  • total_rows                  = {total_rows}")

if dq_passed:
    print("\n✅ All critical DQ checks passed. Safe to load to Silver.")
else:
    print("\n⚠️ Warning: Non‑critical DQ issues detected; review recommended.")


StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 18, Finished, Available, Finished)


 Data Quality Report:
  • null_favorite_sports        = 0
  • invalid_marketing_opt_in    = 0
  • empty_odds_format           = 0
  • duplicate_user_ids          = 0
  • total_rows                  = 4238

✅ All critical DQ checks passed. Safe to load to Silver.


#### **Step 8: Data Load to Silver Lakehouse Layer**
This step writes the cleaned user_preferences data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/oracle/cleaned/cleaned_user_preferences.

Prepare Columns for Consistency:
- ingestion_date: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [17]:
# Define the path for saving the cleaned user preferences in the Silver layer
silver_path = "Files/cdp_silver/oracle/cleaned/cleaned_user_preferences"

# Add a new column to track when each record was ingested (today’s date)
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "ingestion_date", current_date()
)

# Generate a load timestamp to help with record tracking and versioning
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_user_preferences = df_cleaned_user_preferences.withColumn(
    "load_timestamp", lit(load_ts)
)

# Check if the destination Delta table already exists
if DeltaTable.isDeltaTable(spark, silver_path):
    # ✅ If it exists, perform an UPSERT using Delta Merge
    delta_table = DeltaTable.forPath(spark, silver_path)

    (
        delta_table.alias("target")
        .merge(
            df_cleaned_user_preferences.alias("source"),
            "target.user_id = source.user_id"  # Match records by user_id
        )
        .whenMatchedUpdateAll()              # Update matching records
        .whenNotMatchedInsertAll()           # Insert new records
        .execute()
    )

else:
    # If table doesn’t exist, create it and write all records (initial load)
    (
        df_cleaned_user_preferences
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ingestion_date")  # Partition by date for better performance
        .save(silver_path)
    )

print(" Cleaned user_preferences loaded to Silver layer.")


StatementMeta(, ca8bfc38-64f8-4a46-9f45-1ca159e07aed, 19, Finished, Available, Finished)

 Cleaned user_preferences loaded to Silver layer.
